# Lecture 11 — Classes and object-oriented design

A class is a type defined in Python code. Each object of a class holds its own data, and the class
holds the functions that act on that data.

Notes: [Lecture 11](https://teaching.borisbolliet.io/c1-2026/materials/class11/notes.pdf) ·
code: [`attractor` at `lecture-11`](https://github.com/borisbolliet/attractor/tree/lecture-11/)

The notebook runs at the root of the `attractor` clone, at the tag `lecture-11`, with the kernel
*Python (attractor 3.13)* that `setup_env.sh` installs:

```bash
cd ~/GitHub/attractor
git fetch --tags --force
git switch --detach lecture-11
source setup_env.sh
```

In [ ]:
import copy, math, os, subprocess, sys, tempfile
from pathlib import Path
import numpy as np

TAG = "lecture-11"
if not Path("trajectory.py").exists():   # outside a clone (Colab, a download): fetch the project
    root = Path(tempfile.mkdtemp()) / "attractor"
    subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "-q", "--depth", "1", "--branch", TAG,
                    "https://github.com/borisbolliet/attractor.git", str(root)], check=True)
    os.chdir(root)
sys.path.insert(0, os.getcwd())
subprocess.run(["git", "describe", "--tags"], capture_output=True, text=True).stdout.strip()

## 1 Objects, classes and attribute lookup

### Classes and instances

A **class** is a type created by a `class` statement. The body of the statement runs once. The names
it binds, functions and constants alike, are the **class attributes**, stored in the dictionary of the
class.

An **instance** is an object of that type, created by calling the class. Each instance has a dictionary
of its own, `vars(obj)`, which holds its **instance attributes**. Right after an instance is created,
the method `__init__` runs on it and fills that dictionary.

A **method** is a function stored in the class. Its first parameter, named `self` by convention,
receives the instance on which the method is called.

In [ ]:
class Point:
    unit = "cm"                      # class attribute
    def __init__(self, x, y):
        self.x, self.y = x, y        # instance attributes
    def norm(self):
        return (self.x**2 + self.y**2) ** 0.5

p = Point(3, 4)
vars(p), p.unit, p.norm()

In [ ]:
"unit" in vars(p), "unit" in vars(Point), "norm" in vars(Point)

The built-in types are classes too. `int` is a class and `3` an instance of it. A module such as
`math` has attributes but is not a class.

In [ ]:
type(3), isinstance(3, int), int("12")

In [ ]:
type(math), isinstance(math, type)

### Attribute lookup and the method resolution order

`class Pixel(Point)` makes `Pixel` a **subclass** of the **base class** `Point`. The **method
resolution order** is the tuple `Pixel.__mro__`: the class, then its bases, ending with `object`.

`obj.name` is evaluated in three steps:

1. Python searches the instance dictionary `vars(obj)`.
2. It then searches the dictionary of each class of the method resolution order, in turn.
3. It returns the value from the first dictionary that holds `name`.

In [ ]:
class Pixel(Point):
    unit = "px"

q = Pixel(3, 4)
[c.__name__ for c in Pixel.__mro__]

In [ ]:
q.unit, q.norm(), "norm" in vars(Pixel)     # unit found in Pixel, norm one class up, in Point

A function found in a class comes back as a **bound method**, a callable that remembers the instance
and passes it as the first argument. `q.norm()` is therefore `Point.norm(q)`.

In [ ]:
type(Point.norm).__name__, type(q.norm).__name__

In [ ]:
q.norm.__func__ is Point.norm, Point.norm(q)

The built-in types follow the same search. `bool` is a subclass of `int` and defines no `__add__`,
so `True + True` runs the method of `int`.

In [ ]:
[c.__name__ for c in bool.__mro__], True + True, "__add__" in vars(bool), "__add__" in vars(int)

An assignment `obj.name = value` does not search: it writes the instance dictionary. The class value
is **shadowed** for that one instance. `del` removes the instance entry and uncovers the class value.

In [ ]:
q.unit = "mm"
q.unit, Pixel.unit, "unit" in vars(q)

In [ ]:
del q.unit
q.unit

### Class and instance attributes

The class body runs once, so a value created there exists once. For a list it is a trap. The call
`a.add(1)` runs in two steps:

1. `self.seen` is a read. It finds no entry in `vars(a)` and reaches the one list of the class.
2. `append` changes that list in place.

`b.seen` reads the same list.

In [ ]:
class Tally:
    seen = []
    def add(self, x):
        self.seen.append(x)

a, b = Tally(), Tally()
a.add(1)
b.seen, "seen" in vars(a), a.seen is Tally.seen

State that belongs to one instance is created in `__init__`:

In [ ]:
class Tally:
    def __init__(self):
        self.seen = []
    def add(self, x):
        self.seen.append(x)

a, b = Tally(), Tally()
a.add(1)
a.seen, b.seen

## 2 Special methods

### Representation, length and indexing

A **special method** has a name that begins and ends with two underscores. The interpreter calls it
when an operator or a built-in function is applied to an instance. `len(obj)` calls
`type(obj).__len__(obj)`, `obj[i]` calls `__getitem__`, `repr(obj)` calls `__repr__`.

In [ ]:
class Vector:
    def __init__(self, x, y): self.x, self.y = x, y
    def __repr__(self): return f"Vector({self.x}, {self.y})"
    def __len__(self): return 2
    def __getitem__(self, i): return (self.x, self.y)[i]

v = Vector(3, 4)
v, len(v), v[1], list(v)      # list(v) calls __getitem__ with 0, 1, 2 until IndexError

In [ ]:
len("abc"), type("abc").__len__("abc")

In [ ]:
class Bare:
    pass

len(Bare())

### Objects that can be called

An instance whose class defines `__call__` is **callable**, like a function. `double(21)` runs
`type(double).__call__(double, 21)`. The object keeps its values in named attributes, where a closure
keeps them in cells.

In [ ]:
class Scale:
    def __init__(self, k): self.k = k
    def __call__(self, x): return self.k * x

double = Scale(2)
double(21), callable(double), list(map(double, [1, 2])), double.k

### Equality with a tolerance

Without `__eq__`, `a == b` tests `a is b`. `__eq__` returns `NotImplemented` for an operand of another
type: Python then tries the other operand's method, and falls back to identity.

In [ ]:
class Approx:
    def __init__(self, v): self.v = v
    def __eq__(self, other):
        if not isinstance(other, Approx):
            return NotImplemented
        return abs(self.v - other.v) <= 1e-9

Approx(1.0) == Approx(1.0 + 1e-12), Approx(1.0) == 1.0

A dictionary or a set files an object by `hash(obj)`. Two objects that compare equal must have the
same hash. The hash inherited from `object` is based on identity, so it would differ for two equal
`Approx` objects. Python therefore sets `__hash__` to `None` in a class that defines `__eq__` and no
`__hash__`: its instances are not **hashable**.

In [ ]:
Approx.__hash__ is None

In [ ]:
{Approx(1.0): "one"}

`==` on two arrays is elementwise and returns an array of booleans, which `if` cannot use:

In [ ]:
a = np.array([1.0, 2.0])
a == a, np.array_equal(a, a), np.allclose(a, a + 1e-12)

In [ ]:
if a == a:
    pass

### Special methods in the course project

The models of [`models.py`](https://github.com/borisbolliet/attractor/blob/lecture-11/models.py) are classes. Their base class `DynamicalSystem` defines `__call__` as a
call of `rhs`, the time derivative of the state, so a loop written for a function accepts a model.
`RK4(h).run` returns a `Trajectory` of [`trajectory.py`](https://github.com/borisbolliet/attractor/blob/lecture-11/trajectory.py), the times and states of one run, which
defines `__repr__`, `__len__`, `__getitem__` and an `__eq__` with `np.allclose`.

In [ ]:
from integrators import RK4, integrate, rk4_step
from models import LotkaVolterra
from trajectory import Trajectory

lv = LotkaVolterra()
tr = RK4(0.05).run(lv, [1.5, 8.0], 0.0, 40.0)
lv, tr, len(tr), tr[1]

In [ ]:
lv(0.0, np.array([1.5, 8.0]))           # __call__ runs lv.rhs

In [ ]:
t, y = integrate(rk4_step, lv, np.array([1.5, 8.0]), 0.0, 40.0, 0.05)
np.array_equal(y, tr.y)                 # the function-based loop of lecture 9 accepts the model

In [ ]:
other = Trajectory(t, y + 1e-9, lv.names)
tr == other, tr == other.every(2), Trajectory.__hash__ is None

## 3 Copies of objects holding arrays

### Shallow and deep copies

`copy.copy(x)` makes a **shallow copy**: a new object whose instance dictionary is a copy of `vars(x)`.
The keys are copied and the values are not, so every attribute of the copy is the same object as the
original's. `copy.deepcopy(x)` makes a **deep copy**: it copies the object and, recursively, every
object reachable from it.

In [ ]:
grid = [[0, 0], [0, 0]]
sh = copy.copy(grid)
sh[0][0] = 1
grid

In [ ]:
copy.copy(grid)[0] is grid[0], copy.deepcopy(grid)[0] is grid[0]

`moved` works in two steps. It makes the shallow copy `new`, whose `x` and `y` are the arrays of `p`.
It then binds a new array, `self.x + dx`, to `new.x`. `new.y` is still the array of `p`.

In [ ]:
class Points:
    def __init__(self, x, y): self.x, self.y = x, y
    def moved(self, dx):
        new = copy.copy(self)
        new.x = self.x + dx           # a new array; y is shared
        return new

p = Points(np.zeros(3), np.zeros(3))
q = p.moved(1.0)
q.x is p.x, q.y is p.y, copy.deepcopy(p).y is p.y

In [ ]:
q.y[0] = 5.0
p.y

### Views inside copies

A basic slice, `start:stop:step`, is a view: a new array object on the memory of the original. A view
bound to an attribute of a copy shares the memory of the original. Element 1 of `p.x[::2]` is element 2
of `p.x`.

In [ ]:
p = Points(np.arange(4.0), np.zeros(4))
s = copy.copy(p)
s.x = p.x[::2]                      # a view
s.x.base is p.x, np.shares_memory(s.x, p.x)

In [ ]:
s.x[1] = -1.0
p.x

A boolean mask copies: the result owns its memory, and its `.base` is `None`. For a view of a view,
`.base` names the first array, the one that owns the memory.

In [ ]:
m = p.x[p.x > 0]
m.base is None, np.shares_memory(m, p.x)

In [ ]:
v = p.x[1:]
v[::2].base is p.x, v[::2].base is v

[`experiments/copies.py`](https://github.com/borisbolliet/attractor/blob/lecture-11/experiments/copies.py) prints, for the methods of `Trajectory` that return a changed run, which
attributes are shared, then writes through each copy:

In [ ]:
%run -m experiments.copies

## 4 Interfaces, dataclasses and composition

### Abstract base classes and Protocols

The **interface** of a class is the set of methods that its callers rely on. An **abstract base class**
marks them with `@abstractmethod`. A subclass that leaves one undefined cannot be instantiated, and the
error names the missing method.

In [ ]:
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

class Shape(ABC):
    @abstractmethod
    def area(self): ...

class Circle(Shape):                 # area not defined
    def __init__(self, r): self.r = r

Circle(1.0)

A **Protocol** declares an interface by its shape alone: a class satisfies it by defining the methods,
without inheriting from it. A runtime-checkable Protocol checks the names only.

In [ ]:
@runtime_checkable
class HasArea(Protocol):
    def area(self): ...

class Square:                        # no base class
    def __init__(self, s): self.s = s
    def area(self): return self.s ** 2

sq = Square(3)
isinstance(sq, HasArea), isinstance(sq, Shape), isinstance(3, HasArea)

In [ ]:
class Fake:
    area = 3                          # a number, not a method

isinstance(Fake(), HasArea)

In [ ]:
Fake().area()

In [ ]:
from collections.abc import Sized
isinstance("abc", Sized), isinstance(3, Sized)

### Dataclasses

`@dataclass` receives a class and writes `__init__`, `__repr__` and `__eq__` from the annotated names
of the body, the **fields**. With `frozen=True` an assignment to a field raises, and `__hash__` is
generated from the fields.

In [ ]:
from dataclasses import dataclass, field, replace

@dataclass(frozen=True)
class Point:
    x: float
    y: float = 0.0

p = Point(1.0)
p, p == Point(1.0, 0.0), hash(p) == hash(Point(1.0))

In [ ]:
p.x = 2.0

In [ ]:
replace(p, x=2.0)

Without the decorator, a class with the same `__init__` compares by identity:

In [ ]:
class PlainPoint:
    def __init__(self, x): self.x = x

PlainPoint(1.0) == PlainPoint(1.0)

`field(default_factory=list)` gives each instance its own list. A bare list as default is refused
when the decorator builds the class.

In [ ]:
@dataclass
class Path:
    points: list = field(default_factory=list)

Path().points is Path().points

In [ ]:
@dataclass
class Bad:
    points: list = []

### Composition over inheritance

**Inheritance** says "is a": the subclass inherits every attribute of its base, wanted or not.
**Composition** says "has a": an object holds another and forwards calls to it. A stack returns the
last element it received first. `Stack` holds a list and offers only `push` and `pop`; a subclass of
`list` would also inherit `insert` and `sort`, which break that order.

In [ ]:
class Stack:
    def __init__(self): self.items = []          # has a list
    def push(self, x): self.items.append(x)
    def pop(self): return self.items.pop()

s = Stack(); s.push(1); s.push(2)
s.pop(), hasattr(s, "insert"), hasattr(list, "insert")

In [ ]:
class ListStack(list):                            # is a list
    push = list.append

ls = ListStack(); ls.push(2); ls.push(1); ls.sort()
ls.pop()                                          # 2, although 1 was pushed last

### One interface, several classes

Code that calls a method by name runs with any object whose class defines that method:
**polymorphism**. `total_area` adds the areas of squares, rectangles, and any class written later
that defines `area`.

In [ ]:
def total_area(shapes):
    return sum(s.area() for s in shapes)

class Rectangle:
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

total_area([Square(2), Rectangle(2, 3)])

In [ ]:
[len(x) for x in ("abc", [1, 2], {"k": 0})]

In the project, `RK4.run` calls `system(t, y)` and reads `system.names`, so one integrator runs the
three model classes of [`models.py`](https://github.com/borisbolliet/attractor/blob/lecture-11/models.py):

In [ ]:
from models import Goodwin, RosenzweigMacArthur

runs = [(LotkaVolterra(), (1.5, 8.0)), (RosenzweigMacArthur(), (1.5, 0.5)),
        (Goodwin(0.03, 0.05, 0.5, 0.45), (0.95, 0.65))]
for system, y0 in runs:
    tr = RK4(0.05).run(system, y0, 0.0, 300.0)
    print(f"{type(system).__name__:20} {tr.period():5.1f}  {tr.names}")

The exercises, with their solutions, are at the end of the notes; their scripts are in
[`exercises/lecture11/`](https://github.com/borisbolliet/attractor/tree/lecture-11/exercises/lecture11).